# Part 1.2 – Schema design and insertion
Clean the dataset based on the findings in Part 1.1, then insert it into MongoDB.

## Schema
We use two collections.

**`movies`**: one document per movie, with `_id` = TMDB id. Credits and keywords are embedded, because:
- each movie has exactly one credits row and one keywords row, and they are always read together with the movie,
- their size is bounded (the largest movie fits easily within the 16 MB document limit),
- most Part 2 questions (directors, actor pairs, genre breadth, top-billed cast, director–actor pairs) can then be answered with `$unwind`/`$group` and no joins.

People, genres, companies and keywords are not split into separate collections. Their `{id, name}` pairs are repeated in every movie they appear in. This duplication is deliberate:
- the dataset is never updated after insertion, so repeated names cannot become inconsistent,
- each entry keeps its TMDB `id`, which works as a reference without a separate collection, so two people with the same name are still kept apart,
- fields such as `character`, `order`, `job` and `department` describe a person in one specific movie. A separate `people` collection would therefore also need a join collection for these fields, and the Part 2 queries would need a `$lookup` for every movie.

**`ratings`**: one document per (user, movie) rating, referencing `movies._id` through `movie_id`. Ratings are not embedded in `movies` because there are 26M of them and the arrays would be unbounded. The most-rated movie has 91,921 ratings.

```
movies {
  _id: 862, title, original_title, original_language, overview, tagline, status,
  release_date: ISODate, runtime, budget, revenue, popularity, vote_average, vote_count,
  adult, video, imdb_id, homepage, poster_path,
  genres: [{id, name}], production_companies: [{id, name}],
  production_countries: [{iso_3166_1, name}], spoken_languages: [{iso_639_1, name}],
  belongs_to_collection: {id, name},
  keywords: [{id, name}],
  cast: [{id, name, character, order, gender: "female" | "male"}],   // sorted by order, one entry per role
  crew: [{id, name, job, department}]
}
ratings { _id: ObjectId, user_id, movie_id, rating, timestamp: ISODate }
```

### Cleaning rules (from Part 1.1)
- Drop the 3 malformed rows (non-numeric `id`) and the 3 truncated rows just before them (a line break in `overview` split each movie in two). The truncated rows have no `title`, `release_date`, `popularity` or votes, so these 3 movies are left out.
- Duplicate ids: metadata duplicates differ only in `popularity` and `vote_count`, so we keep the row with the highest `vote_count`. Credits duplicates differ only in crew and keyword duplicates are identical, so we keep the first row.
- `budget`, `revenue` and `runtime` = 0 mean unknown, so the field is left out. `vote_average` is left out when `vote_count` = 0.
- Missing scalar values are left out of the document instead of being stored as null. Array fields are always present (empty list if there is no data), so `$unwind`, `$size` and `$arrayElemAt` behave the same for every document.
- Nested Python-literal strings are parsed. Unused sub-fields are dropped, such as poster paths in collections, `credit_id`, `cast_id` and `profile_path`.
- An actor listed more than once in the same movie is kept as several cast entries, one per role. Each entry is a separate TMDB credit, and many are different characters played by the same actor, so they are not merged. Part 2 counts each actor once per movie where this matters (questions 2, 3, 6 and 8). No movie lists the same director twice.
- Cast `gender` 0 (unknown) is left out. 1/2 are stored as `"female"`/`"male"`.
- Dates are stored as BSON dates: `release_date`, and the rating `timestamp` converted from Unix seconds.
- Ratings are mapped from MovieLens `movieId` to TMDB id via `links.csv`. Ratings with no match in `movies` are dropped. Some TMDB ids have two MovieLens ids, so a user can have two ratings for the same movie. In that case only the latest rating is kept, and a unique index on (`user_id`, `movie_id`) enforces this.

In [1]:
# Imports and path to the dataset folder
import ast
import math
from pprint import pprint

import pandas as pd
from DbConnector import DbConnector

DATA = "dataset/"

# Parse a Python-literal string into a list/dict; None if missing or invalid
def parse(text):
    try:
        return ast.literal_eval(text) if isinstance(text, str) else None
    except (ValueError, SyntaxError):
        return None

# Return v if it is a list, otherwise an empty list (so array fields are always present)
def as_list(value):
    return value if isinstance(value, list) else []

# True if a value is missing (None, NaT or NaN)
def missing(value):
    return value is None or value is pd.NaT or (isinstance(value, float) and math.isnan(value))

# Connect to MongoDB and drop old collections so the notebook can be re-run from scratch
conn = DbConnector()
db = conn.db
for collection in ["movies", "ratings"]:
    db[collection].drop()

You are connected to the database: movies_db
-----------------------------------------------



## movies_metadata.csv

In [2]:
# Load metadata and drop corrupt rows (non-numeric id)
movies = pd.read_csv(DATA + "movies_metadata.csv", low_memory=False)
movies = movies[pd.to_numeric(movies["id"], errors="coerce").notna()].copy()
movies["id"] = movies["id"].astype(int)
# Drop the truncated rows just before each corrupt row (no title, release date, popularity or votes)
movies = movies[movies["title"].notna()]

# Convert numeric columns
for col in ["budget", "revenue", "runtime", "popularity", "vote_average", "vote_count"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")
# Keep the row with the most votes per duplicate id (duplicates only differ in popularity and vote_count)
movies = movies.sort_values("vote_count", ascending=False, kind="stable").drop_duplicates("id").sort_index()
# Budget, revenue and runtime 0 mean unknown, so set them to NaN
for col in ["budget", "revenue", "runtime"]:
    movies.loc[movies[col] == 0, col] = float("nan")
# vote_average is meaningless without votes
movies.loc[movies["vote_count"] == 0, "vote_average"] = float("nan")
# Parse dates and convert adult/video to booleans
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
for col in ["adult", "video"]:
    movies[col] = movies[col].astype(str) == "True"
# Strip whitespace from text fields and treat empty strings as missing
for col in ["title", "original_title", "overview", "tagline"]:
    movies[col] = movies[col].str.strip().replace("", float("nan"))
# Parse nested fields from strings into lists/dicts
for col in ["genres", "production_companies", "production_countries", "spoken_languages", "belongs_to_collection"]:
    movies[col] = movies[col].apply(parse)

# Print the cleaned shape and check that all runtimes are whole minutes (so they can be stored as ints)
print(movies.shape)
print("Non-integer runtimes:", (movies["runtime"].dropna() % 1 != 0).sum())

(45430, 24)
Non-integer runtimes: 0


## credits.csv and keywords.csv

In [3]:
# Map TMDB gender codes to strings (0 = unknown is left out)
GENDER = {1: "female", 2: "male"}

# Keep only the cast fields we need, adding gender only if it is known
def cast_entry(person):
    entry = {"id": person["id"], "name": person["name"], "character": person["character"], "order": person["order"]}
    if person["gender"] in GENDER:
        entry["gender"] = GENDER[person["gender"]]
    return entry

# Build the cast sorted by billing order (an actor with several roles keeps one entry per role)
def cast_list(cast):
    return [cast_entry(person) for person in sorted(as_list(parse(cast)), key=lambda person: person["order"])]

# Load credits and keywords, keeping the first row per duplicate id
credits = pd.read_csv(DATA + "credits.csv").drop_duplicates("id")
keywords = pd.read_csv(DATA + "keywords.csv").drop_duplicates("id")

# Build lookups from movie id -> cast, crew and keywords
cast_by_id = {movie_id: cast_list(cast) for movie_id, cast in zip(credits["id"], credits["cast"])}
crew_by_id = {movie_id: [{key: person[key] for key in ("id", "name", "job", "department")} for person in as_list(parse(crew))]
              for movie_id, crew in zip(credits["id"], credits["crew"])}
keywords_by_id = {movie_id: as_list(parse(keyword_list)) for movie_id, keyword_list in zip(keywords["id"], keywords["keywords"])}

# Check that (almost) every movie has credits and keywords
print("Credits:", len(cast_by_id), "| keywords:", len(keywords_by_id))
print("Movies without credits:", (~movies["id"].isin(cast_by_id.keys())).sum(),
      "| without keywords:", (~movies["id"].isin(keywords_by_id.keys())).sum())

Credits: 45432 | keywords: 45432
Movies without credits: 1 | without keywords: 1


## Build and insert movie documents

In [4]:
# Scalar fields to copy into the movie document, and which of them are stored as ints
SCALARS = ["title", "original_title", "original_language", "overview", "tagline", "status",
           "release_date", "runtime", "budget", "revenue", "popularity", "vote_average", "vote_count",
           "adult", "video", "imdb_id", "homepage", "poster_path"]
INTS = {"runtime", "budget", "revenue", "vote_count"}

# Build one movie document: skip missing scalars, convert types, and embed nested lists, keywords and credits
def movie_doc(movie):
    doc = {"_id": movie["id"]}
    for col in SCALARS:
        value = movie[col]
        if missing(value):
            continue
        if col in INTS:
            value = int(value)
        elif isinstance(value, pd.Timestamp):
            value = value.to_pydatetime()
        doc[col] = value
    for col in ["genres", "production_companies", "production_countries", "spoken_languages"]:
        doc[col] = as_list(movie[col])
    if isinstance(movie["belongs_to_collection"], dict):
        doc["belongs_to_collection"] = {key: movie["belongs_to_collection"][key] for key in ("id", "name")}
    doc["keywords"] = keywords_by_id.get(movie["id"], [])
    doc["cast"] = cast_by_id.get(movie["id"], [])
    doc["crew"] = crew_by_id.get(movie["id"], [])
    return doc

# Insert all movies and create a text index on overview + tagline (used in Part 2)
movie_docs = [movie_doc(movie) for movie in movies.to_dict("records")]
db["movies"].insert_many(movie_docs, ordered=False)
db["movies"].create_index([("overview", "text"), ("tagline", "text")], name="text_overview_tagline")
print("Inserted movies:", db["movies"].count_documents({}))

Inserted movies: 45430


## ratings.csv

In [5]:
# Load links and ratings (compact dtypes to save memory)
links = pd.read_csv(DATA + "links.csv")
ratings = pd.read_csv(DATA + "ratings.csv", dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"})
n_raw = len(ratings)

# Map MovieLens movieId -> TMDB id and drop ratings for movies not in the movies collection
ratings["movie_id"] = ratings["movieId"].map(links.set_index("movieId")["tmdbId"])
ratings = ratings[ratings["movie_id"].isin(movies["id"])]
n_matched = len(ratings)
ratings["movie_id"] = ratings["movie_id"].astype("int32")
# If a user rated the same TMDB movie twice (two MovieLens ids), keep only the latest rating
ratings = ratings.sort_values(["userId", "timestamp"]).drop_duplicates(["userId", "movie_id"], keep="last")

# Report how many ratings were dropped and kept
print("Raw:", n_raw, "| dropped (no matching movie):", n_raw - n_matched,
      "| dropped (duplicate user, movie):", n_matched - len(ratings), "| kept:", len(ratings))
print("Max ratings for one movie:", ratings["movie_id"].value_counts().max())

Raw: 26024289 | dropped (no matching movie): 42722 | dropped (duplicate user, movie): 125 | kept: 25981442
Max ratings for one movie: 91921


In [6]:
# Insert ratings in batches to limit memory use; timestamps are converted from Unix seconds to dates
BATCH = 200_000
for start in range(0, len(ratings), BATCH):
    chunk = ratings.iloc[start:start + BATCH]
    times = pd.to_datetime(chunk["timestamp"], unit="s").dt.to_pydatetime().tolist()
    db["ratings"].insert_many([
        {"user_id": user, "movie_id": movie, "rating": rating, "timestamp": time}
        for user, movie, rating, time in zip(chunk["userId"].tolist(), chunk["movie_id"].tolist(), chunk["rating"].tolist(), times)
    ], ordered=False)
    # Print progress every 25 batches
    if start // BATCH % 25 == 0:
        print(f"{start + len(chunk):,} / {len(ratings):,}")

# A unique (user_id, movie_id) index enforces one rating per user and movie, and also serves lookups by user_id
db["ratings"].create_index([("user_id", 1), ("movie_id", 1)], unique=True)
print("Inserted ratings:", db["ratings"].count_documents({}))

200,000 / 25,981,442
5,200,000 / 25,981,442
10,200,000 / 25,981,442
15,200,000 / 25,981,442
20,200,000 / 25,981,442
25,200,000 / 25,981,442
Inserted ratings: 25981442


## Verify

In [7]:
# Print document count, average size, data size and indexes for each collection
for collection in ["movies", "ratings"]:
    stats = db.command("collstats", collection)
    print(f"{collection}: {stats['count']:,} docs | avg {stats['avgObjSize']:,} B | data {stats['size'] / 1e6:,.0f} MB | indexes {list(stats['indexSizes'])}")

# Show an example movie (Toy Story) with cast/crew/keywords cut to 3 entries, and one rating
doc = db["movies"].find_one({"_id": 862})
doc.update({key: doc[key][:3] for key in ["cast", "crew", "keywords"]})
pprint(doc)
pprint(db["ratings"].find_one())

movies: 45,430 docs | avg 3,211 B | data 146 MB | indexes ['_id_', 'text_overview_tagline']
ratings: 25,981,442 docs | avg 84 B | data 2,182 MB | indexes ['_id_', 'user_id_1_movie_id_1']
{'_id': 862,
 'adult': False,
 'belongs_to_collection': {'id': 10194, 'name': 'Toy Story Collection'},
 'budget': 30000000,
 'cast': [{'character': 'Woody (voice)',
           'gender': 'male',
           'id': 31,
           'name': 'Tom Hanks',
           'order': 0},
          {'character': 'Buzz Lightyear (voice)',
           'gender': 'male',
           'id': 12898,
           'name': 'Tim Allen',
           'order': 1},
          {'character': 'Mr. Potato Head (voice)',
           'gender': 'male',
           'id': 7167,
           'name': 'Don Rickles',
           'order': 2}],
 'crew': [{'department': 'Directing',
           'id': 7879,
           'job': 'Director',
           'name': 'John Lasseter'},
          {'department': 'Writing',
           'id': 12891,
           'job': 'Screenplay',
 

In [8]:
# Sanity checks: counts of movies with key fields, movies with an actor in several roles, and the largest document size (must be < 16 MB)
print("Movies with budget and revenue:", db["movies"].count_documents({"budget": {"$exists": True}, "revenue": {"$exists": True}}))
print("Movies with vote_average:", db["movies"].count_documents({"vote_average": {"$exists": True}}))
print("Movies with a director:", db["movies"].count_documents({"crew.job": "Director"}))
print("Movies in a collection:", db["movies"].count_documents({"belongs_to_collection": {"$exists": True}}))
print("Movies with an actor in several roles:", db["movies"].count_documents({"$expr": {"$ne": [{"$size": "$cast"}, {"$size": {"$setUnion": ["$cast.id"]}}]}}))
print("Largest movie document (bytes):", next(db["movies"].aggregate([
    {"$group": {"_id": None, "max": {"$max": {"$bsonSize": "$$ROOT"}}}}]))["max"])

Movies with budget and revenue: 5375
Movies with vote_average: 42534
Movies with a director: 44542
Movies in a collection: 4487
Movies with an actor in several roles: 951
Largest movie document (bytes): 43379


In [9]:
# Close the database connection
conn.close_connection()


-----------------------------------------------
Connection to movies_db-db is closed


## Summary
- `movies`: 45,430 documents with credits and keywords embedded, plus a text index on `overview` + `tagline` for question 7 in part 2.
- `ratings`: about 26M documents that reference `movies._id`, with a unique index on (`user_id`, `movie_id`).
- Unknown values (0 budget/revenue/runtime, 0-vote averages, unknown gender, missing dates) are left out instead of stored as 0, so Part 2 aggregations (`$median`, `$avg`) ignore them automatically.
- An actor with several roles in a movie has one cast entry per role, so Part 2 must count distinct actor ids per movie (e.g. `$setUnion` on `cast.id`) when it counts movies or pairs per actor.